In [0]:
%sql
SELECT
    f.company_code,
    f.fiscal_year,
    f.vendor_id,
    v.vendor_name,

    SUM(f.debit_amount - f.credit_amount)
        AS annual_vendor_expense,

    COUNT(DISTINCT CONCAT(
        f.company_code, '-',
        f.fiscal_year, '-',
        f.document_number
    )) AS journal_entry_count

FROM ask_sarthak.gold.fact_gl_transactions f

JOIN ask_sarthak.gold.dim_gl_account a
    ON f.gl_account = a.gl_account

JOIN ask_sarthak.gold.dim_vendor v
    ON f.vendor_id = v.vendor_id

WHERE a.account_type = 'EXPENSE'
  AND f.vendor_id IS NOT NULL

GROUP BY
    f.company_code,
    f.fiscal_year,
    f.vendor_id,
    v.vendor_name

ORDER BY
    f.company_code,
    f.fiscal_year,
    annual_vendor_expense DESC;


Why are we doing this?

This gives us a historical spending baseline for each vendor and company. We'll use it to identify unusual spending patterns.

Run it and share the output. Next, we'll implement statistical anomaly detection using average and standard deviation, with safeguards against false positives.


2. Next step — Monthly vendor anomaly detection
Annual totals can hide unusual transactions. We'll now calculate monthly vendor expenditure and compare it with that vendor's previous three observed months.

In [0]:
%sql
CREATE OR REPLACE VIEW
ask_sarthak.analytics.vw_vendor_monthly_anomalies
AS

WITH monthly_expense AS (
    SELECT
        f.company_code,
        f.vendor_id,
        v.vendor_name,
        DATE_TRUNC('MONTH', f.posting_date) AS expense_month,

        SUM(f.debit_amount - f.credit_amount)
            AS monthly_expense

    FROM ask_sarthak.gold.fact_gl_transactions f

    JOIN ask_sarthak.gold.dim_gl_account a
        ON f.gl_account = a.gl_account

    JOIN ask_sarthak.gold.dim_vendor v
        ON f.vendor_id = v.vendor_id

    WHERE a.account_type = 'EXPENSE'
      AND f.vendor_id IS NOT NULL

    GROUP BY
        f.company_code,
        f.vendor_id,
        v.vendor_name,
        DATE_TRUNC('MONTH', f.posting_date)
),

historical_baseline AS (
    SELECT
        *,

        COUNT(*) OVER (
            PARTITION BY company_code, vendor_id
            ORDER BY expense_month
            ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
        ) AS previous_month_count,

        AVG(monthly_expense) OVER (
            PARTITION BY company_code, vendor_id
            ORDER BY expense_month
            ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
        ) AS historical_avg,

        STDDEV_SAMP(monthly_expense) OVER (
            PARTITION BY company_code, vendor_id
            ORDER BY expense_month
            ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
        ) AS historical_stddev

    FROM monthly_expense
)

SELECT
    *,
    CASE
        WHEN previous_month_count < 3
            THEN 'INSUFFICIENT_HISTORY'

        WHEN historical_stddev IS NULL
          OR historical_stddev = 0
            THEN 'REVIEW_REQUIRED'

        WHEN monthly_expense >
             historical_avg + (3 * historical_stddev)
            THEN 'HIGH'

        WHEN monthly_expense >
             historical_avg + (2 * historical_stddev)
            THEN 'MEDIUM'

        ELSE 'NORMAL'
    END AS risk_level

FROM historical_baseline;


What this implements:
- Calculates monthly vendor expenses.
- Looks at the previous three observed months for that vendor and company.
- Calculates historical average and standard deviation.
- Flags unusually high spending.
- Avoids classifying vendors with insufficient history as normal.

Step 8 — Inspect anomaly results
Run this SQL query:

In [0]:
%sql
SELECT
    company_code,
    vendor_id,
    vendor_name,
    expense_month,
    monthly_expense,
    historical_avg,
    historical_stddev,
    previous_month_count,
    risk_level
FROM ask_sarthak.analytics.vw_vendor_monthly_anomalies
WHERE risk_level IN ('HIGH', 'MEDIUM')
ORDER BY
    CASE risk_level
        WHEN 'HIGH' THEN 1
        WHEN 'MEDIUM' THEN 2
    END,
    monthly_expense DESC
LIMIT 50;

What we're checking: Whether any vendor's monthly expense is significantly higher than its recent historical spending.

Understanding the first anomaly
Vendor V0297 — Company 2000
HIGH RISK
May 2024 expense
₹5,51,104.80

Historical average
₹4,83,447.97

Historical standard deviation

₹5,607.90

Previous observations

3


Expense exceeded the historical average by approximately ₹67,657, which is more than 3 standard deviations.

That explains the HIGH classification.

Important observation: Your synthetic data has unusually low historical standard deviations compared with the expense values. This may cause many records to be flagged. We should investigate before treating these as genuine financial anomalies.
2. Next practical exercise — Investigate one vendor
Run this SQL to inspect Vendor V0297's monthly expense history and understand why May 2024 was flagged.

In [0]:
%sql
SELECT
    company_code,
    vendor_id,
    vendor_name,
    expense_month,
    monthly_expense,
    historical_avg,
    historical_stddev,
    risk_level
FROM ask_sarthak.analytics.vw_vendor_monthly_anomalies
WHERE company_code = '2000'
  AND vendor_id = 'V0297'
ORDER BY expense_month;

In [0]:
%sql

CREATE OR REPLACE VIEW
ask_sarthak.analytics.vw_vendor_monthly_anomalies
AS

WITH monthly_expense AS (
    SELECT
        f.company_code,
        f.vendor_id,
        v.vendor_name,
        CAST(DATE_TRUNC('MONTH', f.posting_date) AS DATE)
            AS expense_month,
        SUM(f.debit_amount - f.credit_amount)
            AS monthly_expense
    FROM ask_sarthak.gold.fact_gl_transactions f
    JOIN ask_sarthak.gold.dim_gl_account a
        ON f.gl_account = a.gl_account
    JOIN ask_sarthak.gold.dim_vendor v
        ON f.vendor_id = v.vendor_id
    WHERE a.account_type = 'EXPENSE'
      AND f.vendor_id IS NOT NULL
    GROUP BY
        f.company_code,
        f.vendor_id,
        v.vendor_name,
        CAST(DATE_TRUNC('MONTH', f.posting_date) AS DATE)
),

date_bounds AS (
    SELECT
        MIN(expense_month) AS min_month,
        MAX(expense_month) AS max_month
    FROM monthly_expense
),

vendor_pairs AS (
    SELECT DISTINCT
        company_code,
        vendor_id,
        vendor_name
    FROM monthly_expense
),

calendar AS (
    SELECT
        vp.company_code,
        vp.vendor_id,
        vp.vendor_name,
        cal.expense_month
    FROM vendor_pairs vp
    CROSS JOIN date_bounds b
    LATERAL VIEW EXPLODE(
        SEQUENCE(
            b.min_month,
            b.max_month,
            INTERVAL 1 MONTH
        )
    ) cal AS expense_month
),

complete_months AS (
    SELECT
        c.company_code,
        c.vendor_id,
        c.vendor_name,
        c.expense_month,
        m.monthly_expense
    FROM calendar c
    LEFT JOIN monthly_expense m
        ON c.company_code = m.company_code
       AND c.vendor_id = m.vendor_id
       AND c.expense_month = m.expense_month
),

historical_baseline AS (
    SELECT
        *,
        COUNT(monthly_expense) OVER (
            PARTITION BY company_code, vendor_id
            ORDER BY expense_month
            ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
        ) AS previous_month_count,

        AVG(monthly_expense) OVER (
            PARTITION BY company_code, vendor_id
            ORDER BY expense_month
            ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
        ) AS historical_avg,

        STDDEV_SAMP(monthly_expense) OVER (
            PARTITION BY company_code, vendor_id
            ORDER BY expense_month
            ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
        ) AS historical_stddev

    FROM complete_months
)

SELECT
    *,
    CASE
        WHEN monthly_expense IS NULL
            THEN 'NO_ACTIVITY'

        WHEN previous_month_count < 3
            THEN 'INSUFFICIENT_HISTORY'

        WHEN historical_stddev IS NULL
          OR historical_stddev = 0
            THEN 'REVIEW_REQUIRED'

        WHEN monthly_expense >
             historical_avg + (3 * historical_stddev)
            THEN 'HIGH'

        WHEN monthly_expense >
             historical_avg + (2 * historical_stddev)
            THEN 'MEDIUM'

        ELSE 'NORMAL'
    END AS risk_level

FROM historical_baseline;


In [0]:
%sql
SELECT
    expense_month,
    monthly_expense,
    historical_avg,
    historical_stddev,
    previous_month_count,
    risk_level
FROM ask_sarthak.analytics.vw_vendor_monthly_anomalies
WHERE company_code = '2000'
  AND vendor_id = 'V0297'
ORDER BY expense_month;


2. Next exercise — Overall anomaly risk summary
Now let's evaluate the detection engine across all vendors and companies.

Run:

In [0]:
%sql
SELECT
    risk_level,
    COUNT(*) AS vendor_month_count,
    COUNT(DISTINCT vendor_id) AS unique_vendors,
    ROUND(
        SUM(COALESCE(monthly_expense, 0)), 2
    ) AS total_expense
FROM ask_sarthak.analytics.vw_vendor_monthly_anomalies
GROUP BY risk_level
ORDER BY
    CASE risk_level
        WHEN 'HIGH' THEN 1
        WHEN 'MEDIUM' THEN 2
        WHEN 'REVIEW_REQUIRED' THEN 3
        WHEN 'NORMAL' THEN 4
        WHEN 'INSUFFICIENT_HISTORY' THEN 5
        WHEN 'NO_ACTIVITY' THEN 6
    END;


3. Next practical exercise — Build explainable anomaly alerts
Before changing our statistical model, let's make the existing HIGH and MEDIUM alerts explainable.

Create a new SQL view:


In [0]:
%sql
CREATE OR REPLACE VIEW
ask_sarthak.analytics.vw_financial_anomaly_alerts
AS

SELECT
    company_code,
    vendor_id,
    vendor_name,
    expense_month,
    monthly_expense,
    historical_avg,
    historical_stddev,
    risk_level,

    ROUND(
        monthly_expense - historical_avg, 2
    ) AS expense_increase,

    ROUND(
        100.0 * (monthly_expense - historical_avg)
        / NULLIF(historical_avg, 0), 2
    ) AS percentage_increase,

    ROUND(
        (monthly_expense - historical_avg)
        / NULLIF(historical_stddev, 0), 2
    ) AS z_score,

    CONCAT(
        'Vendor monthly expense is ',
        ROUND(
            100.0 * (monthly_expense - historical_avg)
            / NULLIF(historical_avg, 0), 2
        ),
        '% above the historical average.'
    ) AS alert_reason,

    CASE
        WHEN risk_level = 'HIGH'
            THEN 'Review underlying journal entries and supporting invoices.'
        WHEN risk_level = 'MEDIUM'
            THEN 'Compare current expense with recent vendor transactions.'
    END AS recommended_action

FROM ask_sarthak.analytics.vw_vendor_monthly_anomalies

WHERE risk_level IN ('HIGH', 'MEDIUM');

In [0]:
%sql
SELECT
    company_code,
    vendor_id,
    expense_month,
    monthly_expense,
    expense_increase,
    percentage_increase,
    z_score,
    risk_level,
    alert_reason,
    recommended_action
FROM ask_sarthak.analytics.vw_financial_anomaly_alerts
ORDER BY z_score DESC
LIMIT 10;


Excellent! Your explainable financial anomaly alert view is working correctly. We now have more than just a risk label: we can explain why a vendor was flagged and recommend an investigation action.
1. Evaluate your results
Let's examine the highest-risk record.
Vendor V0153 — Company 3000
HIGH

Monthly expense
₹5,22,637.60

Historical average
₹4,62,583.01

Expense increase

₹60,054.59

Percentage increase

12.98%



Z-score: 13.25

The current expense is 13.25 standard deviations above its historical average. This is a strong statistical alert, although it is not proof of fraud.

2. Important observation
Look at these records:
- V0761: ₹34,680 increase, Z-score 4.56.
- V0641: ₹34,680 increase, Z-score 4.56.
- V0881: ₹34,680 increase, Z-score 4.56.
Repeated values suggest that your synthetic data generator may be producing recurring expense patterns. We should investigate this before treating the alerts as independent real-world anomalies.
3. Next practical exercise — Trace an anomaly to journal entries
This is an important enterprise-level capability.

An AI Finance Controller should not just say:
Vendor expenditure increased by 12.98%.

It should also answer:
Which journal entries caused the increase?

Let's investigate Vendor V0153 for October 2026.

Run this SQL:

In [0]:
%sql
SELECT
    f.company_code,
    f.fiscal_year,
    f.document_number,
    f.line_item,
    f.posting_date,
    f.vendor_id,
    v.vendor_name,
    f.gl_account,
    a.account_name,
    a.account_type,
    f.debit_amount,
    f.credit_amount,
    f.debit_amount - f.credit_amount AS net_expense
FROM ask_sarthak.gold.fact_gl_transactions f
JOIN ask_sarthak.gold.dim_gl_account a
    ON f.gl_account = a.gl_account
JOIN ask_sarthak.gold.dim_vendor v
    ON f.vendor_id = v.vendor_id
WHERE f.company_code = '3000'
  AND f.vendor_id = 'V0153'
  AND f.posting_date >= DATE '2026-10-01'
  AND f.posting_date < DATE '2026-11-01'
  AND a.account_type = 'EXPENSE'
ORDER BY net_expense DESC;


4. What we're building

- Ask Sarthak — Financial Investigation Flow
- 1. Detect unusual vendor expense
- 2. Generate HIGH/MEDIUM risk alert
- 3. Identify contributing journal entries — current task
- 4. Explain the anomaly using Genie
- 5. Recommend an investigation action

Run the journal-entry query and share the output. We'll use it to connect the anomaly detection engine to transaction-level financial evidence.

2. Next exercise — Reconcile the anomaly with journal entries

In [0]:
%sql
SELECT
    f.company_code,
    f.vendor_id,

    COUNT(DISTINCT f.document_number)
        AS document_count,

    COUNT(*) AS expense_line_count,

    ROUND(
        SUM(f.debit_amount - f.credit_amount), 2
    ) AS journal_entry_expense,

    a.monthly_expense AS anomaly_report_expense,

    ROUND(
        SUM(f.debit_amount - f.credit_amount)
        - a.monthly_expense, 2
    ) AS reconciliation_difference

FROM ask_sarthak.gold.fact_gl_transactions f

JOIN ask_sarthak.gold.dim_gl_account g
    ON f.gl_account = g.gl_account

JOIN ask_sarthak.analytics.vw_vendor_monthly_anomalies a
    ON f.company_code = a.company_code
   AND f.vendor_id = a.vendor_id
   AND DATE_TRUNC('MONTH', f.posting_date) = a.expense_month

WHERE f.company_code = '3000'
  AND f.vendor_id = 'V0153'
  AND f.posting_date >= DATE '2026-10-01'
  AND f.posting_date < DATE '2026-11-01'
  AND g.account_type = 'EXPENSE'

GROUP BY
    f.company_code,
    f.vendor_id,
    a.monthly_expense;


2. Next step — Automate reconciliation for every alert
Instead of manually checking one vendor, let's create a reusable view that reconciles every HIGH and MEDIUM alert against its underlying journal entries.

In [0]:
%sql
CREATE OR REPLACE VIEW
ask_sarthak.analytics.vw_anomaly_reconciliation
AS

WITH journal_totals AS (
    SELECT
        f.company_code,
        f.vendor_id,
        CAST(DATE_TRUNC('MONTH', f.posting_date) AS DATE)
            AS expense_month,

        COUNT(DISTINCT
            CONCAT_WS(
                '|',
                f.company_code,
                CAST(f.fiscal_year AS STRING),
                f.document_number
            )
        ) AS journal_entry_count,

        COUNT(*) AS expense_line_count,

        SUM(f.debit_amount - f.credit_amount)
            AS journal_entry_expense

    FROM ask_sarthak.gold.fact_gl_transactions f

    JOIN ask_sarthak.gold.dim_gl_account g
        ON f.gl_account = g.gl_account

    WHERE g.account_type = 'EXPENSE'
      AND f.vendor_id IS NOT NULL

    GROUP BY
        f.company_code,
        f.vendor_id,
        CAST(DATE_TRUNC('MONTH', f.posting_date) AS DATE)
)

SELECT
    a.company_code,
    a.vendor_id,
    a.vendor_name,
    a.expense_month,
    a.risk_level,

    a.monthly_expense,
    COALESCE(j.journal_entry_expense, 0)
        AS journal_entry_expense,

    COALESCE(j.journal_entry_count, 0)
        AS journal_entry_count,

    COALESCE(j.expense_line_count, 0)
        AS expense_line_count,

    ROUND(
        a.monthly_expense -
        COALESCE(j.journal_entry_expense, 0), 2
    ) AS reconciliation_difference,

    CASE
        WHEN ABS(
            a.monthly_expense -
            COALESCE(j.journal_entry_expense, 0)
        ) < 0.01
        THEN 'RECONCILED'
        ELSE 'MISMATCH'
    END AS reconciliation_status

FROM ask_sarthak.analytics.vw_financial_anomaly_alerts a

LEFT JOIN journal_totals j
    ON a.company_code = j.company_code
   AND a.vendor_id = j.vendor_id
   AND a.expense_month = j.expense_month;


3. Validate

In [0]:
%sql
SELECT
    risk_level,
    reconciliation_status,
    COUNT(*) AS alert_count
FROM ask_sarthak.analytics.vw_anomaly_reconciliation
GROUP BY
    risk_level,
    reconciliation_status
ORDER BY
    risk_level,
    reconciliation_status;


3. Next step — Prepare the Genie semantic layer
Let's expose the investigation results through a single view that Genie can query.

Create this view:

In [0]:
%sql

CREATE OR REPLACE VIEW
ask_sarthak.analytics.vw_finance_investigation
AS

SELECT
    r.company_code,
    r.vendor_id,
    r.vendor_name,
    r.expense_month,
    r.risk_level,

    r.monthly_expense,
    a.historical_avg,
    a.expense_increase,
    a.percentage_increase,
    a.z_score,
    a.alert_reason,
    a.recommended_action,

    r.journal_entry_count,
    r.expense_line_count,
    r.journal_entry_expense,
    r.reconciliation_difference,
    r.reconciliation_status

FROM ask_sarthak.analytics.vw_anomaly_reconciliation r

JOIN ask_sarthak.analytics.vw_financial_anomaly_alerts a
    ON r.company_code = a.company_code
   AND r.vendor_id = a.vendor_id
   AND r.expense_month = a.expense_month;


In [0]:
%sql
SELECT *
FROM ask_sarthak.analytics.vw_finance_investigation
ORDER BY z_score DESC;